In [2]:

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score


# ─────────────────────────────────────────
# STEP 1: LOAD DATA
# ─────────────────────────────────────────
Housing_path = os.path.join("datasets", "housing")

def load_data(housing_path=Housing_path):
    csv_path = os.path.join(housing_path, "housing.csv")
    return pd.read_csv(csv_path)

housing = load_data()


# ─────────────────────────────────────────
# STEP 2: CREATE income_cat FOR STRATIFIED SPLIT
# ─────────────────────────────────────────
housing["income_cat"] = pd.cut(
    housing["median_income"],
    bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
    labels=[1, 2, 3, 4, 5]
)


# ─────────────────────────────────────────
# STEP 3: STRATIFIED SPLIT
# ─────────────────────────────────────────
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_index, test_index in split.split(housing, housing["income_cat"]):
    strat_train_set = housing.iloc[train_index]
    strat_test_set  = housing.iloc[test_index]


# ─────────────────────────────────────────
# STEP 4: SEPARATE FEATURES AND LABELS
# ─────────────────────────────────────────
housing = strat_train_set.drop("median_house_value", axis=1)
housing_labels = strat_train_set["median_house_value"].copy()


# ─────────────────────────────────────────
# STEP 5: PIPELINES
# ─────────────────────────────────────────
housing_num = housing.drop("ocean_proximity", axis=1)
num_attr    = list(housing_num.columns)
cat_attr    = ["ocean_proximity"]

# Numerical pipeline: fill missing values → scale
num_pipeline = Pipeline([
    ('imputer',    SimpleImputer(strategy="median")),
    ('std_scaler', StandardScaler()),
])

# Full pipeline: numerical + categorical
full_pipeline = ColumnTransformer([
    ('num', num_pipeline, num_attr),
    ('cat', OneHotEncoder(),  cat_attr),
])

housing_prep = full_pipeline.fit_transform(housing)


# ─────────────────────────────────────────
# STEP 6: TRAIN MODEL
# ─────────────────────────────────────────
model = RandomForestRegressor()
model.fit(housing_prep, housing_labels)


# ─────────────────────────────────────────
# STEP 7: EVALUATE
# ─────────────────────────────────────────
housing_predict   = model.predict(housing_prep)
mse               = mean_squared_error(housing_labels, housing_predict)
rmse              = np.sqrt(mse)

scores            = cross_val_score(model, housing_prep, housing_labels,
                                    scoring="neg_mean_squared_error")
cv_rmse_scores    = np.sqrt(-scores)

print(f"RMSE                  = {rmse}")
print(f"Cross-val RMSE (mean) = {cv_rmse_scores.mean()}")
print(f"Cross-val RMSE (std)  = {cv_rmse_scores.std()}")

RMSE                  = 18527.833677207334
Cross-val RMSE (mean) = 49964.77082319867
Cross-val RMSE (std)  = 815.2425379519391
